# 4장. NumPy 기초: 배열로 생각하기

| 항목 | 내용 |
|---|---|
| 예상 시간 | 5시간 |
| 선수 장 | 2장, 3장 |
| 이 장의 결과물 | 문서-단어 카운트 행렬, 루프 없는 행 정규화·코사인 유사도·상위 k개 코드 |
| 과제 연결 | 제약 사항 "TF-IDF와 코사인 유사도는 NumPy로 직접 구현" |

**학습 목표**
- 배열의 `shape`, `dtype`을 읽고 원하는 모양의 배열을 만들 수 있다.
- 단어 목록에서 문서-단어 카운트 행렬을 루프 없이 만들 수 있다.
- `axis=0`과 `axis=1`이 각각 무엇을 합치는지 설명할 수 있다.
- 브로드캐스팅 규칙으로 "행마다 나누기", "열마다 곱하기"를 할 수 있다.
- 0으로 나누기, `log(0)` 같은 함정을 피할 수 있다.
- 점수 배열에서 상위 k개 문서를 뽑고, 두 행렬의 오차를 측정할 수 있다.

이 장은 이 자료에서 **가장 중요한 도구 장**입니다. TF-IDF 구현(13장)과 검색(15장)의 코드는 거의 전부 이 장의 문법 조합입니다.

In [1]:
# 미니 코퍼스: 모든 장에서 함께 쓰는 문서 5개 (0장 0.5절 참고)
MINI_DOCS = [
    "The rocket launched into space!",
    "NASA launched a new space telescope.",
    "The pitcher threw a fast ball.",
    "Our team won the baseball game.",
    "Astronauts watched the baseball game in space.",
]
MINI_LABELS = ["space", "space", "baseball", "baseball", "space"]

import numpy as np

np.set_printoptions(precision=3, suppress=True, linewidth=120)

STOP = {"the", "a", "into", "our", "in", "new"}
FILTERED = [[t for t in doc.lower().strip(".!").split() if t not in STOP] for doc in MINI_DOCS]
VOCAB = sorted({t for doc in FILTERED for t in doc})
WORD_TO_INDEX = {w: i for i, w in enumerate(VOCAB)}
print("어휘", len(VOCAB), "개:", VOCAB)

어휘 15 개: ['astronauts', 'ball', 'baseball', 'fast', 'game', 'launched', 'nasa', 'pitcher', 'rocket', 'space', 'team', 'telescope', 'threw', 'watched', 'won']


## 4.1 배열(ndarray) 만들기

NumPy 배열은 **같은 자료형의 숫자를 격자 모양으로** 담은 객체입니다. 파이썬 리스트와 달리 원소 전체에 한 번에 연산할 수 있고, 훨씬 빠릅니다.

| 속성 | 의미 | 예: 문서 5개 × 어휘 15개 행렬 |
|---|---|---|
| `shape` | 각 축의 길이를 담은 튜플 | `(5, 15)` |
| `ndim` | 축의 개수(차원) | `2` |
| `size` | 전체 원소 개수 | `75` |
| `dtype` | 원소 자료형 | `int64`, `float64` 등 |

In [2]:
v = np.array([3, 4])
M = np.array([[1, 2, 3],
              [4, 5, 6]])

for name, arr in [("v", v), ("M", M)]:
    print(f"{name}: shape={arr.shape}, ndim={arr.ndim}, size={arr.size}, dtype={arr.dtype}")

print("\nzeros  :\n", np.zeros((2, 3)))           # 기본 자료형은 float64
print("ones   :", np.ones(3, dtype=int))
print("arange :", np.arange(0, 10, 2))
print("full   :", np.full(3, 0.5))
print("reshape:\n", np.arange(6).reshape(2, 3))

v: shape=(2,), ndim=1, size=2, dtype=int64
M: shape=(2, 3), ndim=2, size=6, dtype=int64

zeros  :
 [[0. 0. 0.]
 [0. 0. 0.]]
ones   : [1 1 1]
arange : [0 2 4 6 8]
full   : [0.5 0.5 0.5]
reshape:
 [[0 1 2]
 [3 4 5]]


- 원소 1개짜리 축도 축입니다. `shape=(5,)`는 1차원(길이 5), `shape=(5, 1)`은 2차원(5행 1열)입니다. 둘은 **다르게 동작**하므로 4.5절에서 다시 봅니다.
- 자료형은 `astype`으로 바꿉니다. 정수 배열에 소수를 **제자리 연산**(`/=`)으로 넣으려 하면 에러가 납니다. TF-IDF는 소수 계산이므로 카운트 행렬을 만든 뒤 `float64`로 바꿔 씁니다.

In [3]:
counts = np.array([2, 0, 1])
print(counts / 2)                 # 새 배열을 만들면 자동으로 float이 된다
try:
    counts /= 2                   # 정수 배열 자체에 소수를 넣으려 하면 실패
except Exception as e:
    print(type(e).__name__, ":", str(e).splitlines()[0])
print(counts.astype(np.float64) / 2)

[1.  0.  0.5]
UFuncTypeError : Cannot cast ufunc 'divide' output from dtype('float64') to dtype('int64') with casting rule 'same_kind'
[1.  0.  0.5]


## 4.2 인덱싱과 문서-단어 행렬 만들기

| 문법 | 결과 |
|---|---|
| `X[i]` | i번째 행 (문서 i의 벡터) |
| `X[i, j]` | i행 j열 원소 하나 |
| `X[:, j]` | j번째 열 (단어 j가 각 문서에 나온 횟수) |
| `X[1:3]` | 1, 2번 행 (슬라이싱) |
| `X[[0, 4]]` | 0번과 4번 행 (리스트로 고르기, 팬시 인덱싱) |
| `X[:, [2, 5]]` | 2번과 5번 열 |

### 방법 1: 이중 for 루프
가장 이해하기 쉬운 방법입니다. 빈 행렬을 만들고, 단어가 나올 때마다 해당 칸에 1을 더합니다.

In [4]:
X_loop = np.zeros((len(FILTERED), len(VOCAB)), dtype=np.int64)
for i, doc in enumerate(FILTERED):
    for t in doc:
        X_loop[i, WORD_TO_INDEX[t]] += 1


def show(M, cols=VOCAB, width=6):
    """행렬을 단어 머리글과 함께 출력한다 (단어는 앞 width-1글자만)."""
    print("      " + "".join(f"{c[:width - 1]:>{width}}" for c in cols))
    for i, row in enumerate(M):
        print(f"doc {i} " + "".join(f"{v:>{width}.3g}" for v in row))


show(X_loop)

       astro  ball baseb  fast  game launc  nasa pitch rocke space  team teles threw watch   won
doc 0      0     0     0     0     0     1     0     0     1     1     0     0     0     0     0
doc 1      0     0     0     0     0     1     1     0     0     1     0     1     0     0     0
doc 2      0     1     0     1     0     0     0     1     0     0     0     0     1     0     0
doc 3      0     0     1     0     1     0     0     0     0     0     1     0     0     0     1
doc 4      1     0     1     0     1     0     0     0     0     1     0     0     0     1     0


### 방법 2: 위치 목록을 만들고 한 번에 더하기
"(행 번호, 열 번호)" 목록을 먼저 만든 뒤 해당 칸에 한꺼번에 1을 더합니다. 그런데 직관적인 `X[rows, cols] += 1`에는 **함정**이 있습니다. 같은 칸이 여러 번 나와도 **한 번만** 더해집니다.

In [5]:
idx = np.array([0, 0, 2])          # 0번 칸이 두 번 나온다 (같은 단어가 두 번 나온 문서)

A = np.zeros(3, dtype=int)
A[idx] += 1
print("A[idx] += 1       :", A, "← 0번 칸이 1밖에 안 됐다")

B = np.zeros(3, dtype=int)
np.add.at(B, idx, 1)
print("np.add.at(B, idx) :", B, "← 중복까지 정확히 누적")

A[idx] += 1       : [1 0 1] ← 0번 칸이 1밖에 안 됐다
np.add.at(B, idx) : [2 0 1] ← 중복까지 정확히 누적


`np.add.at`은 중복된 위치도 빠짐없이 누적합니다. 실제 문서에는 같은 단어가 여러 번 나오므로 반드시 이것을 씁니다.

In [6]:
rows = [i for i, doc in enumerate(FILTERED) for _ in doc]
cols = [WORD_TO_INDEX[t] for doc in FILTERED for t in doc]
print("rows:", rows)
print("cols:", cols)

X = np.zeros((len(FILTERED), len(VOCAB)), dtype=np.int64)
np.add.at(X, (rows, cols), 1)
print("루프 결과와 같은가:", np.array_equal(X, X_loop))

rows: [0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 4, 4, 4, 4, 4]
cols: [8, 5, 9, 6, 5, 9, 11, 7, 12, 3, 1, 10, 14, 2, 4, 0, 13, 2, 4, 9]
루프 결과와 같은가: True


### 인덱싱 연습

In [7]:
j = WORD_TO_INDEX["space"]
print("doc 4 벡터       :", X[4])
print("'space' 열       :", X[:, j], "← 어느 문서에 나왔나")
print("doc 0, 4의 space :", X[[0, 4], j])
print("앞 두 문서       :\n", X[:2])

doc 4 벡터       : [1 0 1 0 1 0 0 0 0 1 0 0 0 1 0]
'space' 열       : [1 1 0 0 1] ← 어느 문서에 나왔나
doc 0, 4의 space : [1 1]
앞 두 문서       :
 [[0 0 0 0 0 1 0 0 1 1 0 0 0 0 0]
 [0 0 0 0 0 1 1 0 0 1 0 1 0 0 0]]


### ⚠️ 슬라이싱은 복사가 아니라 "창문(view)"
`X[0]`이나 `X[1:3]`은 원본 데이터를 들여다보는 **창문**입니다. 창문을 통해 값을 바꾸면 원본도 바뀝니다. 원본을 지키려면 `.copy()`를 씁니다.

In [8]:
Y = X.copy()
row = Y[0]          # view
row[:] = 99
print("view를 바꾼 뒤 Y[0]:", Y[0], "← 원본이 바뀌었다")

Z = X.copy()
row = Z[0].copy()   # 복사본
row[:] = 99
print("copy를 바꾼 뒤 Z[0]:", Z[0], "← 원본은 그대로")

view를 바꾼 뒤 Y[0]: [99 99 99 99 99 99 99 99 99 99 99 99 99 99 99] ← 원본이 바뀌었다
copy를 바꾼 뒤 Z[0]: [0 0 0 0 0 1 0 0 1 1 0 0 0 0 0] ← 원본은 그대로


## 4.3 벡터화: 루프 대신 배열 연산

배열끼리의 `+ - * / **`는 **같은 위치 원소끼리** 계산됩니다. 파이썬 for 루프 없이 C로 작성된 내부 코드가 한 번에 처리하므로 훨씬 빠릅니다. 이렇게 루프를 배열 연산으로 바꾸는 것을 **벡터화**라고 합니다.

In [9]:
import time

n = 1_000_000
values = list(range(n))
arr = np.arange(n, dtype=np.float64)

t0 = time.perf_counter()
s_loop = sum(v * v for v in values)
t1 = time.perf_counter()
s_np = (arr * arr).sum()
t2 = time.perf_counter()

print(f"파이썬 루프 : {t1 - t0:.4f}초")
print(f"NumPy       : {t2 - t1:.4f}초  (약 {(t1 - t0) / (t2 - t1):.0f}배 빠름, 실행 환경마다 다름)")
print("결과 같음?  :", s_loop == s_np)

파이썬 루프 : 0.0355초
NumPy       : 0.0015초  (약 24배 빠름, 실행 환경마다 다름)
결과 같음?  : True


과제 데이터는 문서 수천 개 × 어휘 수만 개입니다. 루프로 TF-IDF를 계산하면 몇 분 걸릴 일이 벡터화하면 1초 안에 끝납니다.

## 4.4 축(axis) 연산 ⭐

`sum`, `mean`, `max` 같은 집계 함수에 `axis`를 주면 **그 축을 따라 접어서** 합칩니다.

```
          단어 0  단어 1  단어 2 ...
 doc 0  [   0      1      0   ...]  ─┐
 doc 1  [   0      1      1   ...]   │ axis=1: 행마다 가로로 합침 → 문서별 값 (길이 = 문서 수)
 doc 2  [   1      0      0   ...]  ─┘
           │      │      │
           └──────┴──────┴── axis=0: 열마다 세로로 합침 → 단어별 값 (길이 = 어휘 수)
```

외우는 법: **`axis=0`을 주면 0번 축(행)이 사라집니다.** `(5, 15)`에서 0번 축이 사라지면 결과 shape은 `(15,)`, 즉 단어별 값입니다.

In [10]:
print("X.shape             :", X.shape)
print("단어별 총 등장 횟수 (axis=0):", X.sum(axis=0), X.sum(axis=0).shape)
print("문서별 길이 (axis=1)        :", X.sum(axis=1), X.sum(axis=1).shape)
print("전체 합 (axis 없음)         :", X.sum())

X.shape             : (5, 15)
단어별 총 등장 횟수 (axis=0): [1 1 2 1 2 2 1 1 1 3 1 1 1 1 1] (15,)
문서별 길이 (axis=1)        : [3 4 4 4 5] (5,)
전체 합 (axis 없음)         : 20


### 문서 빈도(DF) 구하기
"단어가 나온 문서 수"는 `X > 0`(나왔으면 True)을 **열 방향으로** 세면 됩니다. True는 1, False는 0으로 더해집니다. 이 한 줄이 12장 IDF 계산의 출발점입니다.

In [11]:
df = (X > 0).sum(axis=0)
for w, d in zip(VOCAB, df):
    if d > 1:
        print(f"{w:<9} DF={d}")

baseball  DF=2
game      DF=2
launched  DF=2
space     DF=3


### keepdims
`keepdims=True`를 주면 접은 축을 길이 1로 **남겨 둡니다**. 결과를 원래 행렬과 다시 계산할 때(다음 절) 필요합니다.

In [12]:
print(X.sum(axis=1).shape)                  # (5,)
print(X.sum(axis=1, keepdims=True).shape)   # (5, 1)

(5,)
(5, 1)


## 4.5 브로드캐스팅 ⭐

모양이 다른 배열끼리 연산하면 NumPy가 작은 쪽을 **늘려서(broadcast)** 맞춥니다. 규칙은 하나입니다.

> 두 shape을 **오른쪽 끝부터** 맞춰 비교해서, 각 축의 길이가 **같거나 둘 중 하나가 1**이면 연산할 수 있다. 1인 쪽이 늘어난다.

| 왼쪽 | 오른쪽 | 결과 | 의미 |
|---|---|---|---|
| `(5, 15)` | `(15,)` | `(5, 15)` ✅ | **열마다** 다른 값을 곱하기 (예: 단어별 IDF 곱하기) |
| `(5, 15)` | `(5,)` | ❌ 에러 | 오른쪽 끝 15와 5가 다름 |
| `(5, 15)` | `(5, 1)` | `(5, 15)` ✅ | **행마다** 다른 값으로 나누기 (예: 문서 길이로 나누기) |

TF-IDF에서 쓰는 두 동작이 정확히 이 표의 첫째, 셋째 줄입니다.

In [13]:
weights = np.arange(1, len(VOCAB) + 1)       # shape (15,): 단어마다 다른 가중치 (IDF 자리)
print("X * weights → shape", (X * weights).shape)
print("doc 0:", (X * weights)[0])

lengths = X.sum(axis=1)                      # shape (5,)
try:
    X / lengths
except ValueError as e:
    print("\nX / lengths → ValueError:", e)

lengths_col = X.sum(axis=1, keepdims=True)   # shape (5, 1)
ratio = X / lengths_col
print("\nX / lengths_col → shape", ratio.shape)
print("각 행의 합:", ratio.sum(axis=1), "← 행마다 길이로 나눠서 합이 1")

X * weights → shape (5, 15)
doc 0: [ 0  0  0  0  0  6  0  0  9 10  0  0  0  0  0]

X / lengths → ValueError: operands could not be broadcast together with shapes (5,15) (5,) 

X / lengths_col → shape (5, 15)
각 행의 합: [1. 1. 1. 1. 1.] ← 행마다 길이로 나눠서 합이 1


`keepdims` 대신 `lengths[:, None]`으로 `(5,)`를 `(5, 1)`로 바꿔도 됩니다. `None`(또는 `np.newaxis`)은 그 자리에 길이 1인 축을 끼워 넣습니다.

## 4.6 불리언 마스크와 0으로 나누기

### 조건으로 고르기
`X > 0`처럼 비교하면 같은 shape의 **True/False 배열(마스크)**이 나옵니다. 마스크로 인덱싱하면 True인 위치만 골라집니다.

In [14]:
VOCAB_ARR = np.array(VOCAB)

print("DF가 2 이상인 단어  :", VOCAB_ARR[df >= 2])
print("doc 4에 나온 단어   :", VOCAB_ARR[X[4] > 0])
print("나왔으면 1, 아니면 0:", np.where(X[4] > 0, 1, 0))

DF가 2 이상인 단어  : ['baseball' 'game' 'launched' 'space']
doc 4에 나온 단어   : ['astronauts' 'baseball' 'game' 'space' 'watched']
나왔으면 1, 아니면 0: [1 0 1 0 1 0 0 0 0 1 0 0 0 1 0]


`np.where(조건, 참일 때 값, 거짓일 때 값)`은 원소마다 if-else를 합니다.

### 0으로 나누기 함정
전처리 후 단어가 하나도 남지 않은 **빈 문서**가 있으면, 그 행의 노름이 0이라서 정규화할 때 `0 / 0 = nan`(숫자가 아님)이 됩니다. `nan`은 이후 모든 계산을 오염시킵니다.

In [15]:
X_with_empty = np.vstack([X, np.zeros(len(VOCAB))])     # 마지막에 빈 문서 추가
norms = np.linalg.norm(X_with_empty, axis=1, keepdims=True)
print("노름:", norms.ravel())

with np.errstate(invalid="ignore"):                     # 경고 출력을 잠시 끈다
    bad = X_with_empty / norms
print("그냥 나누면 마지막 행:", bad[-1][:5], "...")

safe = X_with_empty / np.where(norms == 0, 1, norms)    # 0인 노름은 1로 바꿔서 나눈다
print("안전하게 나누면     :", safe[-1][:5], "...")
print("nan 개수:", np.isnan(bad).sum(), "→", np.isnan(safe).sum())

노름: [1.732 2.    2.    2.    2.236 0.   ]
그냥 나누면 마지막 행: [nan nan nan nan nan] ...
안전하게 나누면     : [0. 0. 0. 0. 0.] ...
nan 개수: 15 → 0


노름이 0이면 1로 나누므로 빈 문서는 0 벡터로 남습니다. 0 벡터는 어떤 쿼리와도 유사도가 0이 되어 자연스럽게 검색 결과에서 빠집니다.

## 4.7 수학 함수

`np.log`, `np.sqrt`, `np.exp` 같은 함수는 **원소마다** 적용됩니다(유니버설 함수).

In [16]:
print("np.log([1, e, 100]) :", np.log([1, np.e, 100]))      # 자연로그
print("np.sqrt([4, 9])     :", np.sqrt([4, 9]))

Xf = X.astype(np.float64)
norm_lib = np.linalg.norm(Xf, axis=1)                        # 행마다 L2 노름
norm_manual = np.sqrt((Xf ** 2).sum(axis=1))                 # 3장 수식 그대로
print("행별 L2 노름        :", norm_lib)
print("수식과 같은가       :", np.allclose(norm_lib, norm_manual))

np.log([1, e, 100]) : [0.    1.    4.605]
np.sqrt([4, 9])     : [2. 3.]
행별 L2 노름        : [1.732 2.    2.    2.    2.236]
수식과 같은가       : True


### ⚠️ `np.where` 안의 `log(0)`
sublinear TF는 $\text{tf} > 0$이면 $1 + \ln(\text{tf})$, 아니면 0입니다. `np.where(X > 0, 1 + np.log(X), 0)`로 쓰고 싶어지지만, `np.where`는 **양쪽 값을 모두 먼저 계산**한 뒤 고르기 때문에 0인 칸에서도 `log(0)`을 계산해 경고가 납니다.

In [17]:
import warnings

C = np.array([[0, 1, 3],
              [2, 0, 10]])

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    tf_where = np.where(C > 0, 1 + np.log(C), 0)
print("np.where 방식 경고:", caught[0].message if caught else "없음")

tf = np.zeros(C.shape)
mask = C > 0
tf[mask] = 1 + np.log(C[mask])          # 0이 아닌 칸만 계산한다
print("마스크 방식 결과  :\n", tf)
print("결과는 같다       :", np.allclose(tf, tf_where))

np.where 방식 경고: divide by zero encountered in log
마스크 방식 결과  :
 [[0.    1.    2.099]
 [1.693 0.    3.303]]
결과는 같다       : True


결과는 같지만 경고가 쌓이면 진짜 문제를 놓칩니다. **마스크로 필요한 칸만 계산**하는 습관을 들이세요.

## 4.8 정렬과 상위 k개 ⭐

| 함수 | 돌려주는 것 |
|---|---|
| `np.sort(a)` | 정렬된 **값** |
| `np.argsort(a)` | 정렬했을 때의 **원래 위치(인덱스)**. 검색에서는 이게 곧 문서 ID |
| `np.argpartition(a, k)` | 앞 k개가 "가장 작은 k개"가 되도록만 대충 나눈 인덱스 (정렬은 안 됨) |

NumPy는 오름차순만 있으므로, 내림차순은 `[::-1]`로 뒤집거나 `-점수`를 정렬합니다.

In [18]:
scores = np.array([0.12, 0.85, 0.40, 0.85, 0.05])

print("np.sort              :", np.sort(scores))
print("argsort (오름차순)    :", np.argsort(scores))
print("argsort[::-1]         :", np.argsort(scores)[::-1], "← 동점인 1, 3이 뒤집혔다")
print("argsort(-s, stable)   :", np.argsort(-scores, kind="stable"), "← 동점은 원래 순서(문서 ID 순)")

np.sort              : [0.05 0.12 0.4  0.85 0.85]
argsort (오름차순)    : [4 0 2 1 3]
argsort[::-1]         : [3 1 2 0 4] ← 동점인 1, 3이 뒤집혔다
argsort(-s, stable)   : [1 3 2 0 4] ← 동점은 원래 순서(문서 ID 순)


`[::-1]`로 뒤집으면 동점 문서의 순서도 뒤집힙니다. 실행할 때마다 결과가 같고 설명하기 쉽도록 **`np.argsort(-scores, kind="stable")`**을 권합니다.

### 문서가 많을 때: argpartition
문서 10만 개 중 상위 5개만 필요한데 전부 정렬하는 것은 낭비입니다. `argpartition`으로 상위 k개 **후보만** 고른 뒤 그 k개만 정렬합니다.

In [19]:
rng = np.random.default_rng(42)
big = rng.random(100_000)
k = 5

t0 = time.perf_counter()
full = np.argsort(-big, kind="stable")[:k]
t1 = time.perf_counter()
cand = np.argpartition(-big, k)[:k]                    # 순서 없는 상위 k개 후보
part = cand[np.argsort(-big[cand], kind="stable")]     # 후보 k개만 정렬
t2 = time.perf_counter()

print("전체 정렬    :", full, f"{t1 - t0:.4f}초")
print("argpartition :", part, f"{t2 - t1:.4f}초")
print("같은 결과?   :", np.array_equal(full, part))

전체 정렬    : [45903 21478  7491 27843 68778] 0.0085초
argpartition : [45903 21478  7491 27843 68778] 0.0009초
같은 결과?   : True


> 점수가 같은 문서가 k번째 경계에 걸쳐 있으면 `argpartition`이 어느 쪽을 고를지는 보장되지 않습니다. 문서 수천 개 규모인 이 과제에서는 `argsort` 하나로 충분하고 더 단순합니다.

## 4.9 두 행렬 비교하기

14장에서 내 TF-IDF 행렬과 sklearn 행렬을 비교할 때 쓰는 도구입니다.

| 코드 | 의미 |
|---|---|
| `np.abs(A - B)` | 원소별 절대 오차 |
| `.max()` / `.mean()` | 최대 오차 / 평균 오차 (과제 검증 로그 항목) |
| `np.array_equal(A, B)` | 완전히 같은가 (실수에는 부적합) |
| `np.allclose(A, B, rtol=0, atol=1e-6)` | 모든 원소에서 $\lvert a - b \rvert \le$ `atol` + `rtol`$\times\lvert b \rvert$ 인가 |

In [20]:
A = rng.random((500, 300))
B = A + rng.normal(0, 1e-10, A.shape)   # 아주 작은 계산 오차만 있는 경우

err = np.abs(A - B)
print(f"최대 오차 {err.max():.2e}, 평균 오차 {err.mean():.2e}")
print("1e-6 이내?:", err.max() <= 1e-6, "| allclose:", np.allclose(A, B, rtol=0, atol=1e-6))

C2 = B.copy()
C2[123, 45] += 1e-3                     # 한 칸에 진짜 버그가 있는 경우
err2 = np.abs(A - C2)
pos = np.unravel_index(err2.argmax(), err2.shape)
print(f"\n최대 오차 {err2.max():.2e} at 위치 {tuple(int(p) for p in pos)}")
print("1e-6 이내?:", err2.max() <= 1e-6)

최대 오차 4.67e-10, 평균 오차 7.98e-11
1e-6 이내?: True | allclose: True

최대 오차 1.00e-03 at 위치 (123, 45)
1e-6 이내?: False


- `np.allclose`의 기본값은 `rtol=1e-05, atol=1e-08`이라 "1e-6 이내"라는 과제 기준과 다릅니다. 기준을 정확히 맞추려면 **최대 절대 오차를 직접 계산**하거나 `rtol=0, atol=1e-6`을 명시합니다.
- 실패하면 `argmax` + `np.unravel_index`로 **어느 문서, 어느 단어**에서 어긋났는지 찾을 수 있습니다. 디버깅의 첫걸음입니다.

## 4.10 메모리 감각

`float64` 원소 하나는 8바이트입니다. 행렬 크기는 `행 × 열 × 8` 바이트입니다.

In [21]:
print("미니 코퍼스 X:", X.shape, X.dtype, X.nbytes, "bytes")
print(f"0인 칸 비율  : {(X == 0).mean():.1%}\n")

for docs, vocab in [(500, 3_247), (7_000, 30_000), (18_000, 100_000)]:
    gb = docs * vocab * 8 / 1024**3
    print(f"{docs:>6,} 문서 × {vocab:>7,} 단어 → {gb:6.2f} GB")

미니 코퍼스 X: (5, 15) int64 600 bytes
0인 칸 비율  : 73.3%

   500 문서 ×   3,247 단어 →   0.01 GB
 7,000 문서 ×  30,000 단어 →   1.56 GB
18,000 문서 × 100,000 단어 →  13.41 GB


- 문서 하나에 나오는 단어는 많아야 수백 개라서 실제 문서-단어 행렬은 **99% 이상이 0**입니다(희소 행렬). 그래도 일반 NumPy 배열은 0도 8바이트씩 저장합니다.
- 그래서 과제에서는 **어휘 크기를 줄입니다**(너무 드문 단어 제거, `min_df`, 11장). 7,000 문서 × 30,000 단어(1.56 GB)를 어휘 5,000개로 줄이면 약 270 MB가 됩니다.
- sklearn은 0이 아닌 칸만 저장하는 **희소 행렬**(scipy sparse)을 돌려줍니다. 비교할 때는 `.toarray()`로 일반 배열로 바꿉니다(14장).
- `float32`로 바꾸면 메모리는 절반이지만 정밀도가 약 1e-7 수준이라, 1e-6 검증에서는 `float64`를 유지합니다.

## 4.11 3장 함수를 NumPy로 바꾸기

3장에서 순수 파이썬으로 만든 함수들이 NumPy로는 이렇게 됩니다.

| 3장 (순수 파이썬) | NumPy |
|---|---|
| `l2_norm(v)` | `np.linalg.norm(v)` |
| `dot(a, b)` | `a @ b` |
| `normalize(v)` | `v / np.linalg.norm(v)` |
| `matvec(X, q)` | `X @ q` |
| 모든 문서 쌍의 유사도 | `Xn @ Xn.T` |

In [22]:
def l2_normalize_rows(M: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(M, axis=1, keepdims=True)
    return M / np.where(norms == 0, 1, norms)


Xn = l2_normalize_rows(X.astype(np.float64))

q = np.zeros(len(VOCAB))
q[[WORD_TO_INDEX["space"], WORD_TO_INDEX["rocket"]]] = 1     # 쿼리 "space rocket"
qn = q / np.linalg.norm(q)

scores = Xn @ qn                                            # 모든 문서와 코사인 유사도
print("문서별 코사인 유사도:", scores)
print("순위                :", np.argsort(-scores, kind="stable"))

print("\n문서 쌍 유사도 행렬 (Xn @ Xn.T):")
print(Xn @ Xn.T)

문서별 코사인 유사도: [0.816 0.354 0.    0.    0.316]
순위                : [0 1 4 2 3]

문서 쌍 유사도 행렬 (Xn @ Xn.T):
[[1.    0.577 0.    0.    0.258]
 [0.577 1.    0.    0.    0.224]
 [0.    0.    1.    0.    0.   ]
 [0.    0.    0.    1.    0.447]
 [0.258 0.224 0.    0.447 1.   ]]


마지막 행렬의 $(i, j)$ 칸은 doc $i$와 doc $j$의 코사인 유사도입니다. 대각선은 자기 자신과의 유사도라 모두 1이고, 행렬이 대각선을 기준으로 대칭입니다. 우주 문서끼리인 doc 0–1이 0.577로 가장 비슷합니다. 두 주제가 섞인 doc 4는 우주 문서(doc 0, 1)와도, 야구 문서(doc 3)와도 0보다 큰 값을 가집니다. 반면 doc 2는 다른 어떤 문서와도 단어를 공유하지 않아 모두 0입니다.

## ✍️ 실습 1: 행 정규화

`for` 없이 행렬의 각 행을 L2 노름으로 나누는 함수 `normalize_rows(M)`를 작성하세요. 빈 행(모두 0)은 0으로 남아야 합니다. `np.linalg.norm`을 쓰지 말고 `sqrt`, `sum`, `keepdims`로 직접 노름을 구해 보세요.

In [23]:
M = np.array([[3.0, 4.0, 0.0],
              [0.0, 0.0, 0.0],
              [1.0, 1.0, 1.0]])

In [24]:
# 여기에 작성하세요

**정답**

In [25]:
def normalize_rows(M: np.ndarray) -> np.ndarray:
    norms = np.sqrt((M ** 2).sum(axis=1, keepdims=True))   # (n, 1)
    return M / np.where(norms == 0, 1, norms)


R = normalize_rows(M)
print(R)
print("행별 길이:", np.sqrt((R ** 2).sum(axis=1)))          # [1, 0, 1]

[[0.6   0.8   0.   ]
 [0.    0.    0.   ]
 [0.577 0.577 0.577]]
행별 길이: [1. 0. 1.]


## ✍️ 실습 2: 상위 k개 문서

점수 배열과 `k`를 받아 `(점수, 문서 ID)` 튜플 리스트를 점수 내림차순으로 돌려주는 `topk(scores, k)`를 작성하세요. 동점이면 문서 ID가 작은 쪽이 앞입니다. `k`가 문서 수보다 크면 문서 수만큼만 돌려줍니다.

In [26]:
s = np.array([0.3, 0.9, 0.1, 0.9, 0.5, 0.0])

In [27]:
# 여기에 작성하세요

**정답**

In [28]:
def topk(scores: np.ndarray, k: int) -> list[tuple[float, int]]:
    order = np.argsort(-scores, kind="stable")[:k]     # 슬라이싱은 길이를 넘어가도 에러가 없다
    return [(float(scores[i]), int(i)) for i in order]


print(topk(s, 3))     # [(0.9, 1), (0.9, 3), (0.5, 4)]
print(topk(s, 10))

[(0.9, 1), (0.9, 3), (0.5, 4)]
[(0.9, 1), (0.9, 3), (0.5, 4), (0.3, 0), (0.1, 2), (0.0, 5)]


`float()`, `int()`로 바꾸는 이유: NumPy 숫자(`np.float64`)는 출력하면 `np.float64(0.9)`처럼 보이고 JSON 저장도 안 되므로, 함수 밖으로 내보낼 때는 파이썬 숫자로 바꿔 줍니다.

## ✍️ 실습 3: IDF 미리 계산해 보기

카운트 행렬 `X`(미니 코퍼스)로 아래 두 값을 루프 없이 계산하세요. 12장에서 배울 식을 미리 써 보는 연습입니다.
1. 단어별 DF (길이 15 배열)
2. smooth IDF: $\text{idf} = \ln\dfrac{1 + N}{1 + \text{df}} + 1$ ($N$ = 문서 수)

그리고 IDF가 가장 큰 단어와 가장 작은 단어를 출력하세요.

In [29]:
# 여기에 작성하세요

**정답**

In [30]:
N = X.shape[0]
df = (X > 0).sum(axis=0)
idf = np.log((1 + N) / (1 + df)) + 1

for w, d, v in zip(VOCAB, df, idf):
    print(f"{w:<11} df={d}  idf={v:.3f}")
print("\nIDF 최대:", VOCAB_ARR[idf == idf.max()])
print("IDF 최소:", VOCAB_ARR[idf == idf.min()])

astronauts  df=1  idf=2.099
ball        df=1  idf=2.099
baseball    df=2  idf=1.693
fast        df=1  idf=2.099
game        df=2  idf=1.693
launched    df=2  idf=1.693
nasa        df=1  idf=2.099
pitcher     df=1  idf=2.099
rocket      df=1  idf=2.099
space       df=3  idf=1.405
team        df=1  idf=2.099
telescope   df=1  idf=2.099
threw       df=1  idf=2.099
watched     df=1  idf=2.099
won         df=1  idf=2.099

IDF 최대: ['astronauts' 'ball' 'fast' 'nasa' 'pitcher' 'rocket' 'team' 'telescope' 'threw' 'watched' 'won']
IDF 최소: ['space']


가장 많은 문서(3개)에 나온 `space`의 IDF가 가장 작고, 한 문서에만 나온 단어들의 IDF가 가장 큽니다. "드문 단어일수록 중요하다"는 IDF의 아이디어가 숫자로 보입니다.

## 🧐 퀴즈

**Q1.** shape이 `(500, 3247)`인 행렬 `X`에서 `X.sum(axis=0)`과 `X.sum(axis=1)`의 shape과 의미는?
<details><summary>▶ 정답 보기</summary>

`axis=0`은 `(3247,)`로 단어별 합계(모든 문서에서 그 단어가 나온 총 횟수), `axis=1`은 `(500,)`로 문서별 합계(문서 길이)입니다.
</details>

**Q2.** `X / X.sum(axis=1)`이 에러가 나는 이유와 고치는 방법 두 가지는?
<details><summary>▶ 정답 보기</summary>

`(500, 3247)`과 `(500,)`를 오른쪽 끝부터 맞추면 3247과 500이 달라서 브로드캐스팅이 안 됩니다. `X.sum(axis=1, keepdims=True)` 또는 `X.sum(axis=1)[:, None]`으로 `(500, 1)`을 만들면 행마다 나눌 수 있습니다.
</details>

**Q3.** 같은 단어가 두 번 나온 문서에서 `X[rows, cols] += 1`로 카운트하면 어떤 문제가 생기나요?
<details><summary>▶ 정답 보기</summary>

같은 위치가 여러 번 나와도 한 번만 더해져서 개수가 1로 나옵니다. `np.add.at(X, (rows, cols), 1)`을 써야 중복이 모두 누적됩니다.
</details>

**Q4.** 점수 배열의 내림차순 순위를 `np.argsort(scores)[::-1]`로 구하면 무엇이 문제인가요?
<details><summary>▶ 정답 보기</summary>

동점인 문서들의 순서까지 뒤집혀서, 문서 ID가 큰 쪽이 먼저 나옵니다. `np.argsort(-scores, kind="stable")`을 쓰면 동점일 때 원래 순서(ID 오름차순)가 유지됩니다.
</details>

**Q5.** 빈 문서가 있을 때 L2 정규화를 그대로 하면 무슨 일이 생기고, 어떻게 막나요?
<details><summary>▶ 정답 보기</summary>

노름이 0이라 `0/0 = nan`이 생기고, 이후 유사도 계산까지 nan으로 오염됩니다. `np.where(norms == 0, 1, norms)`로 0인 노름을 1로 바꿔 나누면 빈 문서는 0 벡터로 남습니다.
</details>

**Q6.** 과제의 "오차 1e-6 이내" 검증에 `np.allclose(A, B)`를 기본값 그대로 쓰면 안 되는 이유는?
<details><summary>▶ 정답 보기</summary>

기본값이 `rtol=1e-05, atol=1e-08`이라 상대 오차까지 섞인 다른 기준이 됩니다. `np.abs(A - B).max() <= 1e-6`으로 직접 계산하거나 `rtol=0, atol=1e-6`을 명시해야 과제 기준과 같아집니다.
</details>

## ✅ 체크포인트
- [ ] 단어 목록에서 `np.add.at`으로 카운트 행렬을 만들 수 있다.
- [ ] `axis=0`(단어별)과 `axis=1`(문서별)의 결과 shape을 바로 말할 수 있다.
- [ ] `keepdims`나 `[:, None]`으로 행마다 나누기를 할 수 있다.
- [ ] 0으로 나누기와 `log(0)`을 피하는 코드를 쓸 수 있다.
- [ ] `argsort`로 동점까지 고려한 상위 k개를 뽑을 수 있다.
- [ ] 두 행렬의 최대/평균 오차를 계산하고 어긋난 위치를 찾을 수 있다.

## 🔗 과제 연결과 다음 장
- `np.add.at`(카운트), `axis` 합계(DF), 브로드캐스팅(IDF 곱하기, 정규화)은 13장 `NumpyTfidf`의 핵심 코드가 됩니다.
- `Xn @ qn`과 `argsort`는 15–16장 검색 함수에, 오차 측정은 14장 검증 로그에 그대로 쓰입니다.
- **5장. Pandas 기초**에서는 실제 데이터셋을 표로 불러와 결측치와 중복을 정리합니다.